* **Dataset:** UCI Online Retail II (2 sheets, 1,067,371 rows, Dec 2009 se Dec 2011)
* **Goal:** Identify and document data quality problems before any cleaning
* **Scope:** This notebook only checks the data. All cleaning is done in `03_data_cleaning`.
* **Raw data:** `retail_raw.csv` is never modified. The `InvoiceDate` conversion and the `missing_id` helper column exist only in memory.

#### Importing library

In [1]:
import pandas as pd

#### Loading the data

In [2]:
df = pd.read_csv('retail_raw.csv')

In [3]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [4]:
## checking the shape
df.shape

(1067371, 9)

In [5]:
## checking the data types
df.dtypes

Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
Sales          float64
dtype: object

#### Check 1: Missing Values

In [6]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
Sales          541910
dtype: int64

In [7]:
df.isnull().mean()*100

Invoice         0.000000
StockCode       0.000000
Description     0.410541
Quantity        0.000000
InvoiceDate     0.000000
Price           0.000000
Customer ID    22.766873
Country         0.000000
Sales          50.770538
dtype: float64

* **Finding:** 243,007 rows (22.77%) have no Customer ID and 4,382 rows (0.41%) have no Description. All other columns are complete. `Sales` shows 50.77% nulls only because the second sheet has no `Sales` column, so it is not a data quality problem (see Check 12).
* **Impact:** Customer-level analysis (RFM, retention, churn) is not possible for rows without a Customer ID. Rows without a Description cannot be labelled by product name.
* **Planned action:** Keep rows without a Customer ID for revenue and product analysis, but exclude them from customer-level analysis. Review the rows without a Description before deciding (see Check 3, Check 4 and Check 7).

#### Check 2: Duplicates

In [8]:
df.duplicated().sum()

np.int64(12133)

In [9]:
df.duplicated(subset = ['Invoice','StockCode','Description','Quantity','InvoiceDate','Price','Customer ID','Country']).sum()

np.int64(34335)

In [10]:
df[(df['Invoice']=='489517')& (df['StockCode']=='21912')]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,3.75
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,3.75
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,3.75


- **Finding:** 34,335 rows (3.22%) are exact duplicates when `Sales` is excluded from the comparison. Only 12,133 were flagged when `Sales` was included, because the second sheet has no `Sales` column (NaN), so rows repeated across sheets looked different. The two sheets overlap between 1 Dec 2010 and 9 Dec 2010, so the same transactions appear in both. Of these 34,335 duplicates, 22,523 are copies of the same transactions in the overlap week (the overlap rows are identical in both sheets), and the other 11,812 occur within a single sheet.
- **Impact:** Duplicates would inflate revenue, quantity sold and order counts, especially for December 2010, since the overlap week is counted twice.
- **Planned action:** Remove the cross-sheet duplicates from the overlap week, since they are clearly redundant. Investigate the within-sheet duplicates before dropping them, because the same item can legitimately appear more than once on one invoice (for example, StockCode 21912 appears three times on invoice 489517). Document the final assumption in the cleaning notebook.

#### Check 3: Quantity

In [11]:
df[df['Quantity']<0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,-35.4
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,-9.9
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,-17.0
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia,-12.6
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,-35.4
...,...,...,...,...,...,...,...,...,...
1065910,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397.0,United Kingdom,NaN
1067002,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498.0,United Kingdom,NaN
1067176,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311.0,United Kingdom,NaN
1067177,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315.0,United Kingdom,NaN


In [12]:
df[df['Quantity']==0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales


In [13]:
df[df['Invoice'].str.startswith('C')]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,-35.4
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,-9.9
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,-17.0
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia,-12.6
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,-35.4
...,...,...,...,...,...,...,...,...,...
1065910,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397.0,United Kingdom,NaN
1067002,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498.0,United Kingdom,NaN
1067176,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311.0,United Kingdom,NaN
1067177,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315.0,United Kingdom,NaN


In [14]:
df[(df['Invoice'].str.startswith('C'))& (df['Quantity']<0)]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,-35.4
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,-9.9
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,-17.0
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia,-12.6
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,-35.4
...,...,...,...,...,...,...,...,...,...
1065910,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397.0,United Kingdom,NaN
1067002,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498.0,United Kingdom,NaN
1067176,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311.0,United Kingdom,NaN
1067177,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315.0,United Kingdom,NaN


In [15]:
df[(~df['Invoice'].str.startswith('C'))& (df['Quantity']<0)]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom,0.0
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom,0.0
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom,0.0
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom,0.0
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom,0.0
...,...,...,...,...,...,...,...,...,...
1060794,581210,23395,check,-26,2011-12-07 18:36:00,0.0,NaN,United Kingdom,NaN
1060796,581212,22578,lost,-1050,2011-12-07 18:38:00,0.0,NaN,United Kingdom,NaN
1060797,581213,22576,check,-30,2011-12-07 18:38:00,0.0,NaN,United Kingdom,NaN
1062371,581226,23090,missing,-338,2011-12-08 09:56:00,0.0,NaN,United Kingdom,NaN


In [16]:
df[(df['Invoice'].str.startswith('C')) & (df['Quantity'] >= 0)]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales
76799,C496350,M,Manual,1,2010-02-01 08:24:00,373.57,NaN,United Kingdom,373.57


* **Finding:** No rows have zero Quantity. 22,950 rows have negative Quantity. Of these, 19,493 are cancelled invoices (Invoice starts with 'C'), and 3,457 are non-'C' rows with Price = 0, no Customer ID, and descriptions like "check", "lost" or "damaged", which look like stock adjustments rather than sales.
* **Impact:** Including them would understate revenue and distort return-rate and customer analysis. The adjustment rows are not customer transactions at all.
* **Planned action:** Keep cancellations in a separate returns table for return-rate and net revenue analysis. Exclude the non-'C' adjustment rows from sales analysis. Investigate the one 'C' invoice with non-negative Quantity.
* **Edge case:** One 'C' invoice (C496350) has positive Quantity (+1) and Price 373.57 with StockCode 'M' ("Manual") and no Customer ID. It is inconsistent with a cancellation and would add to revenue instead of reducing it. It will be excluded along with other non-product rows once StockCodes are reviewed.

#### Check 4: Price

In [17]:
df[df['Price']<0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom,-53594.36
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom,-44031.79
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom,-38925.87
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,NaN
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,NaN


* **Finding:** 5 rows (about 0.0005%) have negative Price, ranging down to -53,594.36. All have Invoice starting with 'A', StockCode 'B', Description "Adjust bad debt", Quantity 1 and no Customer ID. They total roughly -158,676.
* **Impact:** These are accounting adjustments, not product sales. Keeping them would reduce total revenue and distort price statistics such as the minimum and the average.
* **Planned action:** Exclude these rows from sales analysis during cleaning. Review other non-product StockCodes and the 'A' invoice prefix in later checks.

In [18]:
df[df['Price']==0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom,0.0
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom,0.0
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom,0.0
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom,0.0
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom,0.0
...,...,...,...,...,...,...,...,...,...
1062442,581234,72817,NaN,27,2011-12-08 10:33:00,0.0,NaN,United Kingdom,NaN
1063965,581406,46000M,POLYESTER FILLER PAD 45x45cm,240,2011-12-08 13:58:00,0.0,NaN,United Kingdom,NaN
1063966,581406,46000S,POLYESTER FILLER PAD 40x40cm,300,2011-12-08 13:58:00,0.0,NaN,United Kingdom,NaN
1064015,581408,85175,NaN,20,2011-12-08 14:06:00,0.0,NaN,United Kingdom,NaN


In [19]:
df[df['Price']==0].isnull().sum()

Invoice           0
StockCode         0
Description    4382
Quantity          0
InvoiceDate       0
Price             0
Customer ID    6131
Country           0
Sales          2515
dtype: int64

In [20]:
df[(df['Price']==0) & (df['Quantity']<0)]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom,0.0
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom,0.0
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom,0.0
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom,0.0
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom,0.0
...,...,...,...,...,...,...,...,...,...
1060794,581210,23395,check,-26,2011-12-07 18:36:00,0.0,NaN,United Kingdom,NaN
1060796,581212,22578,lost,-1050,2011-12-07 18:38:00,0.0,NaN,United Kingdom,NaN
1060797,581213,22576,check,-30,2011-12-07 18:38:00,0.0,NaN,United Kingdom,NaN
1062371,581226,23090,missing,-338,2011-12-08 09:56:00,0.0,NaN,United Kingdom,NaN


In [21]:
df[(df['Price']==0) & (df['Quantity']<0)].isnull().sum()

Invoice           0
StockCode         0
Description    2689
Quantity          0
InvoiceDate       0
Price             0
Customer ID    3457
Country           0
Sales          1336
dtype: int64

In [22]:
df[(df['Price']==0) & (df['Quantity']>0)]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom,0.0
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom,0.0
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126.0,United Kingdom,0.0
5904,489861,DOT,DOTCOM POSTAGE,1,2009-12-02 14:50:00,0.0,NaN,United Kingdom,0.0
6378,489882,35751C,NaN,12,2009-12-02 16:22:00,0.0,NaN,United Kingdom,0.0
...,...,...,...,...,...,...,...,...,...
1060795,581211,22142,check,14,2011-12-07 18:36:00,0.0,NaN,United Kingdom,NaN
1062442,581234,72817,NaN,27,2011-12-08 10:33:00,0.0,NaN,United Kingdom,NaN
1063965,581406,46000M,POLYESTER FILLER PAD 45x45cm,240,2011-12-08 13:58:00,0.0,NaN,United Kingdom,NaN
1063966,581406,46000S,POLYESTER FILLER PAD 40x40cm,300,2011-12-08 13:58:00,0.0,NaN,United Kingdom,NaN


In [23]:
df[(df['Price']==0) & (df['Quantity']>0)].isnull().sum()

Invoice           0
StockCode         0
Description    1693
Quantity          0
InvoiceDate       0
Price             0
Customer ID    2674
Country           0
Sales          1179
dtype: int64

* **Finding:** 6,202 rows (0.58%) have Price = 0. Of these, 3,457 have negative Quantity and 2,745 have positive Quantity. About 99% of all zero-price rows (6,131) have no Customer ID, and among the 2,745 positive-Quantity rows, 1,693 (61.7%) also have no Description.
* **Impact:** Zero-price rows add units without adding revenue, so they inflate quantity sold, lower average price and distort product-level pricing.
* **Planned action:** Exclude zero-price rows from revenue and price analysis. Review the 71 rows that do have a Customer ID separately before deciding, since their reason (for example free items) is not stated in the data.

#### Check 5: Invoice Patterns

In [24]:
df['Invoice'].str[0].value_counts()

Invoice
5    939382
4    108489
C     19494
A         6
Name: count, dtype: int64

In [25]:
df[df['Invoice'].str.startswith('A')]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom,-53594.36
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom,-44031.79
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom,-38925.87
825443,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom,NaN
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,NaN
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,NaN


* **Finding:** 1,047,871 rows (98.17%) have normal numeric invoices, 19,494 rows (1.83%) start with ‘C’ (cancellations), and 6 rows start with ‘A’. All 6 ‘A’ rows are “Adjust bad debt” entries (StockCode ‘B’, Quantity 1, no Customer ID), with Prices ranging from −53,594.36 to +11,062.06 and a net total of about −147,614.
* **Impact:** These rows are accounting adjustments, not sales. Keeping them would distort total revenue and the price distribution.
* **Planned action:** Exclude all ‘A’ invoices from sales analysis. Treat ‘C’ invoices as returns and analyse them separately from sales.

#### Check 6: StockCode

In [26]:
non_product = df[~df['StockCode'].str.match(r'^\d{5}')]
non_product['StockCode'].value_counts()

StockCode
POST        2122
DOT         1446
M           1421
C2           282
D            177
            ... 
DCGS0027       1
DCGS0016       1
DCGS0006       1
DCGS0044       1
DCGS0036       1
Name: count, Length: 62, dtype: int64

In [27]:
len(non_product)

6093

In [28]:
non_product['StockCode'].nunique()

62

In [29]:
non_product['StockCode'].value_counts().head(30)

StockCode
POST            2122
DOT             1446
M               1421
C2               282
D                177
S                104
BANK CHARGES     102
ADJUST            67
AMAZONFEE         43
DCGS0058          31
gift_0001_20      29
gift_0001_30      29
DCGSSGIRL         25
DCGSSBOY          23
PADS              19
gift_0001_10      16
CRUK              16
TEST001           15
DCGS0076          15
DCGS0003          14
gift_0001_50       8
gift_0001_40       7
DCGS0069           6
B                  6
DCGS0004           5
m                  5
gift_0001_80       4
DCGS0072           4
DCGS0066N          4
gift_0001_70       3
Name: count, dtype: int64

In [30]:
codes = non_product[['StockCode', 'Description']].drop_duplicates(subset=['StockCode'])
codes.head(65)

,StockCode,Description
89,POST,POSTAGE
735,D,Discount
2377,DCGS0058,MISO PRETTY GUM
2378,DCGS0068,DOGS NIGHT COLLAR
2379,DOT,DOTCOM POSTAGE
...,...,...
804712,DCGS0073,ebay
804713,DCGS0071,NaN
804715,DCGS0066P,NaN
804718,DCGS0067,ebay


In [31]:
non_product[non_product['StockCode'].str.startswith('DCGS')]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales
2377,489597,DCGS0058,MISO PRETTY GUM,1,2009-12-01 14:28:00,0.83,NaN,United Kingdom,0.83
2378,489597,DCGS0068,DOGS NIGHT COLLAR,1,2009-12-01 14:28:00,8.65,NaN,United Kingdom,8.65
8371,490074,DCGS0004,HAYNES CAMPER SHOULDER BAG,1,2009-12-03 14:39:00,17.35,NaN,United Kingdom,17.35
8372,490074,DCGS0058,MISO PRETTY GUM,1,2009-12-03 14:39:00,0.83,NaN,United Kingdom,0.83
8373,490074,DCGS0076,SUNJAR LED NIGHT NIGHT LIGHT,1,2009-12-03 14:39:00,16.48,NaN,United Kingdom,16.48
...,...,...,...,...,...,...,...,...,...
843773,564825,DCGSSGIRL,GIRLS PARTY BAG,1,2011-08-30 12:26:00,3.29,NaN,United Kingdom,NaN
891427,568716,DCGSSBOY,BOYS PARTY BAG,2,2011-09-28 16:13:00,3.29,NaN,United Kingdom,NaN
933664,571931,DCGSSGIRL,GIRLS PARTY BAG,1,2011-10-19 16:59:00,3.29,NaN,United Kingdom,NaN
1000063,576840,DCGSSGIRL,GIRLS PARTY BAG,1,2011-11-16 15:23:00,3.29,NaN,United Kingdom,NaN


In [32]:
non_product[non_product['StockCode'].str.startswith('DCGS')].isnull().sum()

Invoice          0
StockCode        0
Description     26
Quantity         0
InvoiceDate      0
Price            0
Customer ID    158
Country          0
Sales           45
dtype: int64

In [33]:
len(non_product[non_product['StockCode'].str.startswith('DCGS')])

158

In [34]:
non_product[non_product['StockCode'].str.startswith('DCGS')]['Description'].value_counts(dropna=False).head(15)

Description
MISO PRETTY  GUM                30
NaN                             26
GIRLS PARTY BAG                 23
BOYS PARTY BAG                  21
SUNJAR LED NIGHT NIGHT LIGHT    15
BOXED GLASS ASHTRAY             13
OOH LA LA DOGS COLLAR            5
ebay                             5
HAYNES CAMPER SHOULDER BAG       4
CAT CAMOUFLAGUE COLLAR           3
NAVY CUDDLES DOG HOODIE          2
DOGS NIGHT COLLAR                2
CAMOUFLAGE DOG COLLAR            2
update                           2
HANDZ-OFF CAR FRESHENER          1
Name: count, dtype: int64

In [35]:
print(codes.to_string())

           StockCode                          Description
89              POST                              POSTAGE
735                D                             Discount
2377        DCGS0058                     MISO PRETTY  GUM
2378        DCGS0068                    DOGS NIGHT COLLAR
2379             DOT                       DOTCOM POSTAGE
2697               M                               Manual
8371        DCGS0004           HAYNES CAMPER SHOULDER BAG
8373        DCGS0076         SUNJAR LED NIGHT NIGHT LIGHT
9292              C2                             CARRIAGE
18410   BANK CHARGES                         Bank Charges
20249       DCGS0003                  BOXED GLASS ASHTRAY
27994        TEST001              This is a test product.
30620   gift_0001_80                                  NaN
30672       DCGS0072               CAT CAMOUFLAGUE COLLAR
32048   gift_0001_20   Dotcomgiftshop Gift Voucher £20.00
33819       DCGS0044              HANDZ-OFF CAR FRESHENER
39411        T

In [36]:
groups = {
    'charges_fees': ['POST', 'DOT', 'C2', 'D', 'BANK CHARGES', 'AMAZONFEE', 'CRUK'],
    'adjustments_test': ['M', 'm', 'B', 'ADJUST', 'ADJUST2', 'TEST001', 'TEST002'],
    'gift_vouchers': ['gift_0001_10', 'gift_0001_20', 'gift_0001_30', 'gift_0001_40',
                      'gift_0001_50', 'gift_0001_60', 'gift_0001_70', 'gift_0001_80',
                      'gift_0001_90'],
    'samples': ['S'],
    'unclear': ['GIFT', 'C3'],
    'other_product_like': ['PADS', 'SP1002']
}

In [37]:
all_codes = []
for codes_list in groups.values():
    all_codes.extend(codes_list)

print(len(all_codes))
print(all_codes)

28
['POST', 'DOT', 'C2', 'D', 'BANK CHARGES', 'AMAZONFEE', 'CRUK', 'M', 'm', 'B', 'ADJUST', 'ADJUST2', 'TEST001', 'TEST002', 'gift_0001_10', 'gift_0001_20', 'gift_0001_30', 'gift_0001_40', 'gift_0001_50', 'gift_0001_60', 'gift_0001_70', 'gift_0001_80', 'gift_0001_90', 'S', 'GIFT', 'C3', 'PADS', 'SP1002']


In [38]:
remaining = non_product[~non_product['StockCode'].isin(all_codes)]

print(len(remaining))
remaining['StockCode'].value_counts().head(15)

158


StockCode
DCGS0058     31
DCGSSGIRL    25
DCGSSBOY     23
DCGS0076     15
DCGS0003     14
DCGS0069      6
DCGS0004      5
DCGS0072      4
DCGS0066N     4
DCGS0068      3
DCGS0070      3
DCGS0037      2
DCGS0062      2
DCGS0075      1
DCGS0074      1
Name: count, dtype: int64

In [39]:
non_product[non_product['StockCode']=='S']

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales
114061,C500305,S,SAMPLES,-1,2010-03-07 10:59:00,73.80,NaN,United Kingdom,-73.80
114083,C500309,S,SAMPLES,-1,2010-03-07 11:09:00,32.03,NaN,United Kingdom,-32.03
133558,C502083,S,SAMPLES,-1,2010-03-22 15:50:00,170.37,NaN,United Kingdom,-170.37
133582,C502088,S,SAMPLES,-1,2010-03-22 16:03:00,259.59,NaN,United Kingdom,-259.59
136253,C502438,S,SAMPLES,-1,2010-03-24 13:11:00,605.18,NaN,United Kingdom,-605.18
...,...,...,...,...,...,...,...,...,...
939608,C572347,S,SAMPLES,-1,2011-10-24 11:03:00,33.05,NaN,United Kingdom,NaN
945127,572849,S,SAMPLES,1,2011-10-26 12:20:00,33.05,NaN,United Kingdom,NaN
1006422,C577330,S,SAMPLES,-1,2011-11-18 14:16:00,2.89,NaN,United Kingdom,NaN
1006423,C577330,S,SAMPLES,-1,2011-11-18 14:16:00,75.00,NaN,United Kingdom,NaN


In [40]:
s_rows = non_product[non_product['StockCode'] == 'S']

print((s_rows['Quantity'] < 0).sum())
print((s_rows['Quantity'] > 0).sum())
print(s_rows['Invoice'].str.startswith('C').sum())
print(s_rows['Customer ID'].isnull().sum())

101
3
101
104


In [41]:
for name, codes_list in groups.items():
    print(name, non_product['StockCode'].isin(codes_list).sum())

print('DCGS', non_product['StockCode'].str.startswith('DCGS').sum())

charges_fees 4188
adjustments_test 1519
gift_vouchers 100
samples 104
unclear 2
other_product_like 22
DCGS 158


* **Finding:** 6,093 rows (0.57%) across 62 StockCodes do not follow the standard 5-digit product code format. Of these, 4,188 are charges/fees (postage, discount, bank charges, commissions), 1,519 are manual, adjustment or test entries, 100 are gift vouchers, 104 are ‘SAMPLES’ rows (101 of them cancellation invoices with no Customer ID), and 180 are non-standard codes (DCGS..., PADS, SP1002) whose descriptions look like real products. Within the DCGS... rows, 125 have a product name and 33 have a blank, “ebay” or “update” description. 2 rows (GIFT, C3) are unclear.
* **Impact:** Including charges, adjustments, vouchers and samples would inflate product sales and distort price and product-level analysis. Removing every non-5-digit code, however, would wrongly drop real product rows.
* **Planned action:** Exclude charges/fees, adjustments/test, gift vouchers and samples from product-sales analysis (postage can be analysed separately). Keep the DCGS... rows with a product description, and review the 33 unclear ones and the 2 unclear codes before deciding.

#### check 7: Description

In [42]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [43]:
valid = df[df['Description'].notnull()]

In [44]:
(valid['Description'].str.strip() != valid['Description']).sum()

np.int64(213035)

In [45]:
df['Description'].isnull().sum()

np.int64(4382)

In [46]:
desc = valid['Description']

In [47]:
leading = desc[desc.str.lstrip() != desc]
print(len(leading))

2624


In [48]:
trailing = desc[desc.str.rstrip() != desc]
print(len(trailing))

211195


In [49]:
print(trailing.head(10).map(repr))

3           'RECORD FRAME 7" SINGLE SIZE '
5             'PINK DOUGHNUT TRINKET POT '
8                              'CAT BOWL '
11    'LUNCHBOX WITH CUTLERY FAIRY CAKES '
12                 'DOOR MAT BLACK FLOCK '
17          'CHRISTMAS CRAFT WHITE FAIRY '
32       'CHRISTMAS CRAFT HEART STOCKING '
33      'PARTY CONE CHRISTMAS DECORATION '
38           'WOODEN BOX ADVENT CALENDAR '
41           'INFLATABLE POLITICAL GLOBE '
Name: Description, dtype: object


In [50]:
print(desc.nunique())
print(desc.str.strip().nunique())

5698
5655


- **Finding:** After cleaning spaces and case, 1,210 of 4,950 StockCodes (about 24.4%) still have more than one distinct Description. In the two top examples (20713, 22734), one real product name accounts for almost all rows (1,372 and 792), while the other descriptions appear only 1 to 5 times each and read like stock or adjustment notes ("FOUND", "MISSING", "WRONGLY MARKED 23343", "AMAZON ADJUSTMENT").
- **Impact:** Using Description as the product label would split one product into several labels and mix notes with product names.
- **Planned action:** Use StockCode as the product key, and assign each StockCode its most frequent Description as the standard product name. Review the note-like descriptions before dropping them.

In [51]:
clean = desc.str.strip()

In [52]:
print(clean.str.contains('  ').sum())

48398


In [53]:
print(clean.str.upper().nunique())

5631


- **Finding:** 48,398 Descriptions (about 4.55%) contain a double space inside the text, and 24 labels differ only by letter case (5,655 distinct after stripping, 5,631 after upper-casing). After collapsing repeated spaces and upper-casing, 5,620 distinct descriptions remain, so 78 of the original 5,698 labels were formatting duplicates.
- **Impact:** These variants split one product across several labels, which affects product-level counts and groupings.
- **Planned action:** Collapse repeated spaces and standardise letter case in the cleaning notebook.

In [54]:
clean2 = clean.str.replace(r'\s+', ' ', regex=True)

print(clean2.nunique())
print(clean2.str.upper().nunique())

5644
5620


In [55]:
valid = df[df['Description'].notnull()].copy()
valid['desc_clean'] = valid['Description'].str.strip().str.replace(r'\s+', ' ', regex=True).str.upper()

counts = valid.groupby('StockCode')['desc_clean'].nunique()
print((counts > 1).sum())
print(counts.sort_values(ascending=False).head(10))

1210
StockCode
20713     8
22734     7
22423     7
23084     7
22719     6
21181     6
21830     6
47566B    6
85175     6
85123A    5
Name: desc_clean, dtype: int64


Finding: After cleaning spaces and case, 1,210 StockCodes still have more than one distinct Description. In the two top examples (20713, 22734), one real product name accounts for almost all rows (1,372 and 792), while the other descriptions are one or two rows each and read like stock or adjustment notes (“FOUND”, “MISSING”, “WRONGLY MARKED 23343”, “AMAZON ADJUSTMENT”).
Impact: Using Description as the product label would split one product into several labels and mix notes with product names.
Planned action: Use StockCode as the product key, and assign each StockCode its most frequent Description as the standard product name. Review the note-like descriptions before dropping them.

In [56]:
print(valid['StockCode'].nunique())

4950


- **Planned action:** Strip leading and trailing spaces from Description in the cleaning notebook, keeping the raw backup unchanged.

#### Checking Customer ID

In [57]:
missing = df[df['Customer ID'].isnull()]

print(len(missing))
print(missing['Invoice'].nunique())
print(df['Invoice'].nunique())
print((missing['Quantity'] * missing['Price']).sum())
print((df['Quantity'] * df['Price']).sum())

243007
8752
53628
2638958.1799999997
19287250.568


In [58]:
df['missing_id'] = df['Customer ID'].isnull()

print(df.groupby('missing_id')[['Quantity', 'Price']].mean())
print(df.groupby('missing_id')['Country'].value_counts().groupby(level=0).head(3))

             Quantity     Price
missing_id                     
False       12.414574  3.676800
True         1.540548  7.948744
missing_id  Country       
False       United Kingdom    741301
            Germany            17624
            EIRE               16195
True        United Kingdom    240029
            EIRE                1671
            Hong Kong            364
Name: count, dtype: int64


In [59]:
print(df.groupby('missing_id')[['Quantity', 'Price']].median())

            Quantity  Price
missing_id                 
False            5.0   1.95
True             1.0   3.29


* **Finding:** 243,007 rows (22.77%) have no Customer ID. They belong to 8,752 of 53,628 invoices (16.32%) and make up about 13.68% of revenue (2.64M of 19.29M, calculated as Quantity × Price). These rows differ from rows with an ID: median Quantity is 1 versus 5, median Price is 3.29 versus 1.95, and 98.8% are from the United Kingdom versus 89.9% for rows with an ID.
* **Impact:** Customer-level analysis (RFM, retention, churn) will cover only about 86% of revenue, and findings about customers should not be generalised to all sales. The missing rows are not a random sample.
* **Planned action:** Keep these rows for overall revenue and product analysis, exclude them from customer-level analysis, and state this coverage gap in the project README.

#### Check 9: Date Range and Sheet Overlap

In [60]:
df['InvoiceDate']= pd.to_datetime(df['InvoiceDate'])

In [61]:
sheet1 = df.iloc[:525461]

In [62]:
sheet1.shape

(525461, 10)

In [63]:
sheet1['InvoiceDate'].min()

Timestamp('2009-12-01 07:45:00')

In [64]:
sheet1['InvoiceDate'].max()

Timestamp('2010-12-09 20:01:00')

In [65]:
sheet2 = df.iloc[525461:]

In [66]:
sheet2['InvoiceDate'].min()

Timestamp('2010-12-01 08:26:00')

In [67]:
sheet2['InvoiceDate'].max()

Timestamp('2011-12-09 12:50:00')

In [68]:
overlap_start = sheet2['InvoiceDate'].min()
overlap_end = sheet1['InvoiceDate'].max()

print(len(sheet1[sheet1['InvoiceDate'] >= overlap_start]))
print(len(sheet2[sheet2['InvoiceDate'] <= overlap_end]))

22523
22523


- **Finding:** Sheet 1 covers 1 Dec 2009 to 9 Dec 2010, and Sheet 2 covers 1 Dec 2010 to 9 Dec 2011. The two sheets overlap between 1 and 9 Dec 2010, and each sheet holds 22,523 rows for that period.
- **Impact:** Transactions from the overlap week appear twice, which inflates revenue and order counts for December 2010 (see Check 2).
- **Planned action:** Remove the duplicated overlap-week rows during cleaning, keeping one copy of each transaction.

#### Check 10: Outliers

In [69]:
df[['Quantity', 'Price']].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999])

,Quantity,Price
0.010,-3.0,0.210
0.050,1.0,0.420
0.250,1.0,1.250
0.500,3.0,2.100
0.750,10.0,4.150
0.950,30.0,9.950
0.990,100.0,18.000
0.999,500.0,217.016


In [70]:
cols = ['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'Customer ID']

print(df.nlargest(10, 'Quantity')[cols])
print(df.nlargest(10, 'Price')[cols])

        Invoice StockCode                         Description  Quantity  \
1065882  581483     23843         PAPER CRAFT , LITTLE BIRDIE     80995   
587080   541431     23166      MEDIUM CERAMIC TOP STORAGE JAR     74215   
90857    497946     37410  BLACK AND WHITE PAISLEY FLOWER MUG     19152   
127166   501534     21099         SET/6 STRAWBERRY PAPER CUPS     12960   
127168   501534     21091         SET/6 WOODLAND PAPER PLATES     12960   
127169   501534     21085           SET/6 WOODLAND PAPER CUPS     12744   
1027583  578841     84826      ASSTD DESIGN 3D PAPER STICKERS     12540   
127167   501534     21092       SET/6 STRAWBERRY PAPER PLATES     12480   
192197   507637     84016          FLAG OF ST GEORGE CAR FLAG     10200   
135027   502269     21984    PACK OF 12 PINK PAISLEY TISSUES      10000   

         Price  Customer ID  
1065882   2.08      16446.0  
587080    1.04      12346.0  
90857     0.10      13902.0  
127166    0.10      13902.0  
127168    0.10      1390

In [71]:
df[(df['StockCode'] == '23843') & (df['Customer ID'] == 16446)]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales,missing_id
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,NaN,False
1065883,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom,NaN,False


In [72]:
df[(df['StockCode'] == '23166') & (df['Customer ID'] == 12346)]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales,missing_id
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom,NaN,False
587085,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,1.04,12346.0,United Kingdom,NaN,False


* **Finding:** 99% of rows have Quantity below 100 and Price below 18, and the 99.9th percentiles are 500 (Quantity) and 217.02 (Price). The maximums are far higher: Quantity 80,995 and Price 38,970. The two largest Quantity values (80,995 and 74,215) were each fully cancelled by the same customer within 12 and 16 minutes (invoices C581484 and C541433). All ten highest Prices belong to non-product StockCodes (Manual, Bank Charges, Amazon Fee), many in order and cancellation pairs at the same Price. Bulk orders also appear, for example five rows on invoice 501534 at a Price of 0.10.
* **Impact:** Extreme values pull averages upward (mean Quantity 9.9 against a median of 3) and would distort revenue totals, average price and chart scales if left in.
* **Planned action:** Do not delete outliers by a fixed cutoff. Remove non-product codes (already covered in Check 6) and handle order and cancellation pairs through the cancellation logic. Keep genuine bulk orders, flagged separately for analysis.

#### Check 11: Country

In [73]:
df['Country'].nunique()

43

In [74]:
df['Country'].value_counts()

Country
United Kingdom          981330
EIRE                     17866
Germany                  17624
France                   14330
Netherlands               5140
Spain                     3811
Switzerland               3189
Belgium                   3123
Portugal                  2620
Australia                 1913
Channel Islands           1664
Italy                     1534
Norway                    1455
Sweden                    1364
Cyprus                    1176
Finland                   1049
Austria                    938
Denmark                    817
Unspecified                756
Greece                     663
Japan                      582
Poland                     535
USA                        535
United Arab Emirates       500
Israel                     371
Hong Kong                  364
Singapore                  346
Malta                      299
Iceland                    253
Canada                     228
Lithuania                  189
RSA                        169


* **Finding:** Country has no missing values and 43 distinct labels. The United Kingdom accounts for 91.94% of rows (981,330). Some labels are not countries: “Unspecified” (756 rows), “European Community” (61) and “West Indies” (54), and “Channel Islands” (1,664) is a territory. Naming is also mixed: EIRE, RSA, USA and Korea appear as short or older forms, while other values are full names.
* **Impact:** Country-level analysis is dominated by the UK. The non-country labels and mixed naming can distort country charts, maps and filters.
* **Planned action:** Keep all rows, standardise country names (for example EIRE to Ireland and RSA to South Africa), and report “Unspecified”, “European Community” and “West Indies” as separate categories rather than guessing their country.

#### Check 12: Sales Column Consistency

In [75]:
s1 = df.iloc[:525461].copy()

calc = (s1['Quantity'] * s1['Price']).round(2)
diff = s1['Sales'].round(2) != calc

print(diff.sum())
print(s1[diff].head(10))

0
Empty DataFrame
Columns: [Invoice, StockCode, Description, Quantity, InvoiceDate, Price, Customer ID, Country, Sales, missing_id]
Index: []


* **Finding:** Sales exists only in the first sheet (525,461 rows). For all of those rows it equals Quantity × Price (0 mismatches after rounding to 2 decimals). It is missing for all 541,910 rows of the second sheet, which is why it shows 50.77% nulls in Check 1.
* **Impact:** The column adds no information, and it is inconsistent across the two sheets, so using it directly would leave half the data without Sales.
* **Planned action:** Drop the original Sales column in cleaning and recalculate it as Quantity × Price for every row.

## Issue Log

| # | Check | Issue | Rows affected | Planned action |
|---|---|---|---|---|
| 1 | Missing values | Customer ID missing; Description missing | Customer ID: 243,007 (22.77%); Description: 4,382 (0.41%) | Keep Customer ID-missing rows for revenue and product analysis, exclude from customer-level analysis; review rows with missing Description |
| 2 | Duplicates | Exact duplicates, mainly from the 1-9 Dec 2010 sheet overlap | 34,335 (3.22%) | Remove overlap-week duplicates; investigate within-sheet duplicates before dropping |
| 3 | Negative Quantity | Cancellations and stock-adjustment-style rows | 22,950 (2.15%): 19,493 'C' invoices, 3,457 non-'C' rows with Price 0 and no Customer ID; 1 'C' invoice with positive Quantity | Keep cancellations as returns, exclude non-'C' adjustment rows from sales, investigate the one inconsistent row |
| 4 | Price | Negative Price (bad debt entries) and zero Price | Negative: 5 rows; zero: 6,202 (0.58%), of which 98.9% have no Customer ID | Exclude from revenue and price analysis; review the 71 zero-price rows that have a Customer ID |
| 5 | Invoice patterns | 'C' (cancellation) and 'A' (bad debt) prefixes | 'C': 19,494 (1.83%); 'A': 6 | Treat 'C' as returns, exclude 'A' from sales analysis |
| 6 | Non-product StockCodes | Charges, adjustments, test rows, gift vouchers, samples, non-standard product codes | 6,093 (0.57%) across 62 codes | Exclude charges, adjustments, vouchers and samples from product sales; keep DCGS product rows; review 33 unclear DCGS rows and 2 unclear codes |
| 7 | Description | Extra spaces, case differences, multiple descriptions per StockCode | 213,035 with leading/trailing spaces (~20%); 48,398 with double spaces; 1,210 of 4,950 StockCodes have more than one description | Strip and standardise spaces and case; use StockCode as product key with its most frequent description |
| 8 | Missing Customer ID | Rows differ from rows with an ID | 243,007 rows, 16.32% of invoices, ~13.68% of revenue | Keep for overall revenue, exclude from customer analysis, document the coverage gap in the README |
| 9 | Date range | Sheets overlap between 1 and 9 Dec 2010 | 22,523 rows in each sheet| Remove the duplicated overlap-week rows, keeping one copy |
| 10 | Outliers | Extreme Quantity and Price values | Max Quantity 80,995 and max Price 38,970 against 99.9th percentiles of 500 and 217; the two largest Quantity values were fully cancelled | Do not drop by a fixed cutoff; remove non-product codes, handle cancellation pairs, flag genuine bulk orders |
| 11 | Country | Non-country labels and mixed naming | 43 labels; "Unspecified" 756, "European Community" 61, "West Indies" 54; UK is 91.94% | Keep all rows, standardise names (e.g. EIRE, RSA), report non-country labels separately |
| 12 | Sales column | Present in sheet 1 only | Missing for 541,910 rows (sheet 2); 0 mismatches in sheet 1 | Drop and recalculate as Quantity x Price for all rows |